# Power Law Raes

In [2]:
using CSV, DataFrames

## Model definition

The Power Law RAES Model is a dynamic random graph generator based on the original RAES model. In this model, each node, at the beginning of the network formation process, assign itself a minimum target degree $d_u$ according to the following rule:
- With probability $p,$ it sets its minimum target degree to a standard value $d,$ that is, $d_u = d.$
- With probability $1-p,$ it sets its minimum target degree according to a power law distribution with exponent $k,$ i.e, $d_u \sim \text{PowerLaw}(k).$

The model is defined by the following parameters:
- $n:$ number of nodes in the network.
- $d:$ standard minimum target degree.
- $c:$ threshold multiplier, used to define the maximum degree tolerance.
- $p:$ Probability of standard behaviour adoption.
- $k:$ Power law exponent.

The graph-generating dynamic is defined as it follows:

- Initialization phase: 
- Starting from an arbitrary initial graph $G_0=(V,E_0)$
    - Each node $u \in V$
        - Set the minimum target degree as the standard one, that is, $d_u = d$ with probability $p,$ otherwise
        - Set the minimum target degree according to a power law distribution with exponent $k.$ 
- At each round $t \in \mathbb{N}$
    - Step $1:$ For each node $u \in V,$ let $N^1_u$ be the set of neighbors of $u$ at the beginning of Step $1.$ If $|N^1_u| < d_u$ then $u$ samples $d_u - |N^1_u|$ nodes from the set $V \setminus N^1_u,$ independently and u.a.r. with replacement and connects to them.
    - Step $2:$ For each node $u \in V,$ let $N^2_u$ be the set of neighbors of $u$ at the beginning of Step $2.$ If $|N^2_u| > c \cdot d_u,$ then $u$ samples $|N^2_u| - (c \cdot d_u)$ neighbors from the set $N^2_u,$ independently and u.a.r. with replacement, and disconnects from them.

In [ ]:
mutable struct plaw_raes_graph
    g::SimpleGraph{Int}
    target_degrees::Vector{Int}
    informed_nodes::BitVector  
    rng::AbstractRNG
    
    function plaw_raes_graph(n::Int, seed::Int, k::Real, d::Int, p_std::Real)
        @assert p_std <= 1
        @assert p_std >= 0 
        local_rng = MersenneTwister(seed)
        target_degs = Vector{Int}(undef, n)
        
        pareto_distr = Pareto(k-1, 1.0)
        for i in 1:n
            r = rand(local_rng)

            if r <= p_std
                target_degs[i] = d
            else 
                target_degs[i] = round(Int, rand(local_rng, pareto_distr))
            end
        end

        new(
            SimpleGraph(n),       
            target_degs,          
            falses(n),          
            MersenneTwister(seed) 
        )
    end
end

function plaw_raes_step_one!(state::plaw_raes_graph) 
    local g = state.g
    local rng = state.rng
    n = nv(g)
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n
        d_u = state.target_degrees[u]
        need = max(0, d_u - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        if length(forbidden) >= n 
            continue 
        end

        count = 0

        while count < need
            target = rand(rng, 1:n) 

            if target ∉ forbidden
                push!(new_edges, (u, target))
                #push!(forbidden, target)  #(without replacement)
                count += 1
            end

        end
    end

    for (u, v) in new_edges
        if !has_edge(g, u, v) 
            add_edge!(g, u, v)
        end
    end
    return g
end

function plaw_raes_step_two!(state::plaw_raes_graph, c::Real) 
    local g = state.g
    local rng = state.rng
    deg_snapshot = degree(g)

    edges_to_remove = Set{Edge{Int}}()
    
    for u in vertices(g)
        d_u = state.target_degrees[u]
        max_deg_u = ceil(Int, d_u * c)
        deg_u = deg_snapshot[u]
        excess = max(0, deg_u - max_deg_u)
        excess == 0 && continue
        
        current_neighbors = neighbors(g, u)
        to_drop = sample(rng, current_neighbors, excess, replace=true)
        
        for v in to_drop
            push!(edges_to_remove, Edge(min(u,v), max(u,v)))
        end
    end
    

    for e in edges_to_remove
        rem_edge!(g, e)
    end
    
    return g
end

# Experiments

Given the model, we are mainly interested in two categories of experiments:
- Structural experiments,  in which we analyze the evolution of the network during the dynamic process and characterize its structure after it reaches stability.
- Diffusion Dynamics, in which we measure the time required for a message, starding from a random node, to reach all the nodes in the network.

## Structural experiments

### Baseline calibration: Parameter selection for the exponent $k.$
In this preliminary experiment, we  investigate the baseline behavior of the model in the pure Power Law regime, that is, $p=0.0.$ We fix $n=2^{15}$ and the standard parameters  $d=4, c=3$ while varying the power law exponent $k.$ In this setting, every node samples its target degree exclusively from the power law distribution.

The goal is to determine how the "heaviness" of the degree distribution tail, controlled by $k,$ influences both the convergence speed of the network and the magnitude of its expansion properties.

For each value of $k \in \{2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5\}$ we execute $100$ independent runs of $100$ rounds each, tracking the evolution of the spectral gap of the network and computing the mean value across the runs at each time step.

![Power law vs Classic](../results/plaw_raes/plot_comparison_time_p0_vs_p1.png)

For $k=2.0,$ the network exhibits a large spectral gap and extremely fast convergence. This suggest the presence of massive hubs, i.e. nodes with a very high minimum target degree, which facilitates the rapid merging of connected components. The presence of those hubs will be investigated in the following experiments.

For $k=2.5$ we notice very small values of the spectral gap, indicating a weakly connected graph structure.

For $k \in \{3, 3.5, 4.0, 4.5, 5\}$ we observe that the spectral gap drops to $0,$ implying that the network becomes disconnected. This shows that without hubs, and lacking the standard uniform mechanism (since $p=0.0$), the local sampling rules fail to connect the graph into a single giant component.

Consequently, this experiment identifies a functional range for the exponent and motivates the choice to fix $k=2.0$ for the subsequent analysis, as it guarantees the strongest connectivity properties in the power-law regime.

### Influence of $p$ on Spectral Expansion and Stability

Building on the previous results, we fix the power-law exponent at $k=2.0$, as this value ensures robust connectivity. In this experiment, we analyze the impact of the probability parameter $p$, which governs the transition between the Power-Law behavior and the Uniform RAES mechanism. We performed simulations for $p \in \{0.1, 0.3, 0.5, 0.7, 0.9\}$, keeping the network size $n=2^{15}$ and the parameters $d=4, c=3$ fixed. As before, we executed 100 independent runs of 100 rounds each, tracking the mean Spectral Gap

![Power law vs Classic](../results/plaw_raes/plot_spectral_gap_by_p.png)

From this experiment, we notice that:
- Regardless of $p,$ all generated networks converge to a stable state with a positive spectral gap, confirming the robustness of the model.
- We observe that decreasing $p$ increases the spectral gap. For $p=0.1,$ the gap stabilizes at a high value, approximately $0.56$. As $p$ increases towards the standard regime (e.g, $p=0.9$) the gap progressively decreases.

To sum this up, we observe that decreasing $p$ significantly increases the spectral gap. This trend suggests that the heavy-tailed target degree distribution, active when $p$ is low, induces a centralized topology structure, characterized by massive hubs. We hypothesize that these nodes act as structural shortcuts, minimizing the diameter and accelerating the mixing process.
In the next section, we will verify the existence of these hubs and quantify the structural transition of the network.

### Topology analysis

In the following experiment we focus on some topological properties of the network.
We use the following metrics:
- Average degree: the average number of connections per node.
- Max degree: The degree of the most connected node in the network.
- Min degree: The degree of the least connected node in the network.
- Percentage below average: The percentage of nodes having a degree lower than the mean.
- Percentage above average: The percentage of nodes having a degree higher than the mean.
- Count nodes with minimum degree: The number of nodes with minimum degree.
- Count nodes with maximum degree: The number of nodes with maximum degree.
- Hub dominance: The percentage of the total edge volume incident to the top $1\%$ nodes, that is, the $1%$ of nodes with the higher degree in the network.

We execute $50$ independent runs of $50$ rounds each, taking the mean of the computed metrics. We do that for $n=2^{15}, d=4, c=3, k=2$ and $p \in \{0.1, 0.2, \ldots , 1.0\}$

In [2]:
df = CSV.read("../results/plaw_raes/structure/topology_stats_k2_extended.csv", DataFrame)
df

Row,p_value,avg_deg,max_deg,min_deg,perc_below_avg,perc_above_avg,count_nodes_min_deg,count_nodes_max_deg,hub_dominance
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,16.3612,27686.0,2.0,82.8815,17.1185,2925.6,1.0,42.3387
2,0.1,15.2811,26058.0,2.0,85.4013,14.5987,1134.7,1.02,39.7484
3,0.3,14.3223,25051.0,1.0,89.1383,10.8617,626.08,1.1,36.287
4,0.5,12.7619,23043.0,1.0,88.4766,11.5234,18.92,1.1,29.9149
5,0.7,11.1421,18784.0,1.0,79.1169,20.8831,14.04,1.0,21.9801
6,0.9,9.23785,12477.0,1.0,64.5837,35.4163,13.96,1.0,10.8571
7,1.0,7.93264,12.0,4.0,44.0392,55.9608,669.14,1614.04,1.51421


![Hub Dominance](../results/plaw_raes/structure/plot_hub_dominance.png)

This graphs shows that, in the case of $p=0.0,$ i.e, every node has its own degree chosen according to a Power law with exponent $k=2.0,$ the top $1\%$ of nodes control over $42 \%$ of the total edges in the network. This shows that the network's connectivity is highly concentrated. The percentage of edges controlled by the top $1\%$ nodes decreases as the probability of a node to assume a standard target degree increases.
We observe that, even at $p=0.9,$ the top nodes still manage approximately $11\%$ of the connection, but, at $p=1.0,$ i.e. the standard model, the dominance collapses to $1.5 \%.$

![Average degs](../results/plaw_raes/structure/plot_avg_degree.png)

We observe a monotonic decline in the average degree as $p$ increases. It starts at $\approx 16.4,$ for $p=0.0$ and drops to $\approx 7.9$ for $p=1.0.$ This show that, when $p<1$ the presence of massive hubs significantly increases the total volume of edges in the network. As $p$ tends to $1,$ the number of nodes with high degree decreases, and the average degrees settles closer to the minimum target value.

![Nodes above average](../results/plaw_raes/structure/plot_inequality.png)

In this graph we see that, for $p=0.0$, roughly $83\%$ of nodes have a degree lower than the average. As $p$ approaches $1.0$, this percentage drops significantly to $\approx 44\%$
This shows that:
- In the "hub scenario", that is, $p < 1,$ the majority of nodes have a degree closer to the minimum, and most of the connections are owned by a few percentage of nodes, as determined before.
- In the "classical scenario", that is, $p=1.0,$ the distribution becomes more symmetric.

We explicitely note that the percentage of nodes below the average degree does not peak at $p=0.0,$ but around $p=0.3.$ This event suggest that the probability $p$ is large enough to force a fraction of nodes to adopt the minimum target degree, that might have otherwise sampled intermediate degrees from the power law distribution. Simultaneously, the remaining power-law component $1-p$ is sufficient enough to generate massive hubs, which keep the global average degree high.

![Max Deg](../results/plaw_raes/structure/plot_max_degree.png)

The maximum degree shows a non-linear behavior. For the entire range of $p$ (from $0.0$ to $0.9$), $d_{max}$ remains high, oscillating between $12,000$ and $27,000$. A phase transition occurs at $p=1.0$, where $d_{max}$ collapses to $12$.

This shows that, even when nodes have only a small probability of adopting a power law target degree, the size of the network guarantees that enough nodes will sample from the tail of the distribution to create massive hubs.

### Assortativity analysis

The previous experiments showed us the presence of hubs, i.e, nodes with a very high minimum target degree, that are connected with most of the nodes in the network. In the following experiment, we're interested to determine if those hubs tends to connects with other nodes that have a relatively high degree, or with nodes that have a small degree.

To do so, we are going to analyze the assortativity of the network, that quantifies the preference of nodes to attach to others with similar degrees. It is measured by the Pearson correlation coefficient $r\in [-1,1]$ of the degrees at either end of an edge.
- If $r>0,$ the network is assortative. It means that high degree nodes tend to connect with other high degree nodes.
- If $r<0,$ the network is disassortative. It means that high degree nodes tend to connect with low degree nodes. This case creates a star-like structure of the graph topology.
- If $r \approx 0,$ then no correlation exists.

In the context of our model, analyzing assortativity allows us to understand how the hubs are organized:
- If the network is disassortative, it means that hubs act as "service providers" for the low-degree nodes, facilitating global connectivity.
- If the network is assortative, the hubs would form a dense core isolated from the other nodes.

Experimental Setup
We measured the assortativity coefficient $r$ for networks generated with $n=2^{15}$, varying $p \in \{0.0, 0.1, \dots, 1.0\}$. We averaged the results over $50$ independent runs of $50$ rounds each.

In [12]:
df = CSV.read("../results/plaw_raes/assortativity_stats_k2.csv", DataFrame)
df

Row,p_value,mean_assortativity,std_assortativity
,Float64,Float64,Float64
1,0.0,-0.222054,0.0680197
2,0.1,-0.184446,0.0642448
3,0.2,-0.188684,0.0524865
4,0.3,-0.193095,0.0747779
5,0.4,-0.141079,0.0433285
6,0.5,-0.150107,0.0570221
7,0.6,-0.129888,0.0539463
8,0.7,-0.100562,0.0332491
9,0.8,-0.0825324,0.043362


![Assortativity](../results/plaw_raes/assortativity_plot.png)

Analyzing the results, we observe that
- For low values of $p,$ the network exhibits significant disassortativity, with $r$ reaching values around $-0.22$ at $p=0.0.$ This confirms that the hubs in the RAES model act as structural bridges, connecting to low degree nodes rather than forming isolated groups. 
- As $p$ increases, the coefficient $r$ monotonically approaches zero. At $p=0.9$ it rises to $\approx -0.05.$ This means that, as the network becomes more uniform, the distinction between high degree nodes and low degree nodes fades. The connection pattern becomes uncorrelated.

### Structural Efficiency and Criticality Analysis
Having analyzed the local connectivity patterns (Assortativity) and the theoretical expansion properties (Spectral Analysis), we now focus on the global transport capabilities of the network. Before simulating dynamic processes like flooding, it is crucial to understand the underlying structural efficiency (Path Lengths) and the distribution of load (Centrality) to identify potential bottlenecks induced by the transition parameter $p$.
To investigate the topological transition from a scale-free structure to a regularized random graph, extensive numerical simulations were performed. The network size was fixed at $n = 2^{15}$ ($32,768$) nodes. The simulation sweeps the control parameter $p,$ that is, the probability of fixed-degree assignment, from $0.0$ to $1.0$ in increments of $0.1.$ For each value of $p$, the following procedure is executed:

- Initialization: A graph is generated mixing scale-free properties (Power law distribution, $k=2.0$) and fixed-degree properties ($d=4$), controlled by $p$.
- Evolution: The network undergoes a dynamic evolution process for $50$ rounds. In each round, edges are added to satisfy degree requirements and removed to maintain the target average degree constraint ($c=3.0$) according to the model definition. 
- Repetition: To ensure statistical robustness, the entire process is repeated for $50$ independent runs for each $p$ value.

Given the computational complexity of calculating path-based metrics on large graphs, exact calculation for all node pairs is infeasible. Therefore, metrics of interest for the model were estimated using a representative random sample of $n_{sample} = 2,000$ source nodes.

To characterize the structural properties of the generated networks, three classes of topological metrics are estimated:

- Global Efficiency (Diameter & APL):
    - Average Path Length (APL): Defined as the average number of steps along the shortest paths for all possible pairs of network nodes. It measures the efficiency of information or mass transport.
    - Diameter: The longest shortest path in the network.

These metrics determine if the network exhibits "Small-World" properties. A low APL indicates a highly efficient network, while a high APL suggests a lattice-like or dispersed structure.

- Community Structure (Clustering Coefficient):
    - Global and Local Clustering: Measures the probability that two neighbors of a node are also neighbors of each other (triadic closure).

High clustering is a hallmark of social and biological networks, indicating strong local communities. A decrease in clustering signals the "randomization" of the topology and the loss of local cohesion.

- Centralization (Maximum Betweenness Centrality):
    - Betweenness Centrality: Quantifies the number of shortest paths passing through a specific node. We focus on the Maximum Betweenness (the score of the most central node in the network).

This metric identifies bottlenecks and network vulnerability. A high maximum betweenness indicates that the network relies heavily on a few "hub" nodes to maintain connectivity. If these nodes fail, the network is prone to fragmentation.

In [3]:
df = CSV.read("../results/plaw_raes/optimized_dynamic_stats.csv", DataFrame)
df

Row,p_value,mean_diameter,std_diameter,mean_apl,std_apl,mean_global_clust,std_global_clust,mean_local_clust,std_local_clust,mean_max_betweenness,std_max_betweenness
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,3.38,0.85452,2.14828,0.233521,0.00215638,0.000532087,0.599842,0.210981,0.45321,0.198132
2,0.1,3.02,0.958102,2.11989,0.258676,0.00181419,0.000286595,0.570527,0.202878,0.520631,0.199821
3,0.2,3.42,1.41551,2.25423,0.340474,0.00150431,0.000216463,0.42138,0.221232,0.506602,0.207969
4,0.3,3.98,1.47759,2.34454,0.347103,0.00137564,0.000182535,0.32874,0.203313,0.502604,0.230624
5,0.4,3.56,1.65566,2.28776,0.362761,0.0011844,0.00014759,0.332533,0.185451,0.584371,0.225355
6,0.5,3.84,1.8556,2.38775,0.443465,0.00110645,0.000131726,0.26982,0.180956,0.570286,0.253343
7,0.6,4.8,1.73793,2.63287,0.491743,0.00103816,9.0803e-5,0.178409,0.162619,0.512853,0.205479
8,0.7,5.02,1.72011,2.74644,0.540517,0.000931243,7.23272e-5,0.141216,0.13947,0.531427,0.242755
9,0.8,5.62,1.85043,3.04784,0.655399,0.0008369,6.79462e-5,0.102956,0.142822,0.547919,0.237736


#### Gobal Efficiency (Diameter and Average Path Length)

As shown in the efficiency analysis, both the Diameter and Average Path Length (APL) exhibit a non-linear growth as $p$ increases.
- When $p < 0.3$: The network maintains a low APL ($\approx 2.1$) and small diameter, characteristic of scale-free networks where "hubs" act as global shortcuts.
- When $p \to 1.0$: As the proportion of fixed-degree nodes dominates, the shortcuts vanish. The topology "unfolds," forcing paths to traverse more hops. The APL more than doubles (reaching $\approx 5.5$), indicating a significant degradation in global communication efficiency.

![Flooding vs P](../results/plaw_raes/diameter_vs_apl.png)

#### Community structure (Clustering coefficient)

The analysis of the Clustering Coefficient reveals a sharp decay in local cohesion.
- At $p=0$, the high local clustering ($\approx 0.6$) reflects a structure rich in triangles and local communities, typical of the generative mechanism for scale-free networks.The 
- introduction of fixed-degree constraints acts as a randomizing force. As $p$ approaches $1.0$, the clustering coefficient drops to near zero, indicating that the network transitions into a random-regular-like graph where local transitivity is statistically negligible.

![Clustering ](../results/plaw_raes/clustering_plot.png)

#### Centralization (Maximum Betweenness Centrality)

The most notable result is observed in the Maximum Betweenness Centrality. Unlike the monotonic trends seen in efficiency and clustering, centralization exhibits a distinct peak behavior in the transition region ($0.3 \le p \le 0.5$).
- Hub Distribution ($p=0$): Load is distributed among several large hubs; while high, the maximum load is manageable.
- Uniform Distribution ($p=1$): The network is democratic; no single node is critical.
- The Critical Phase ($p \approx 0.4$): In this intermediate regime, the network loses enough connectivity to become inefficient but retains a heterogeneous structure. This forces the global traffic to channel through a critically small number of surviving bridges or hubs. This "bottleneck effect" results in an explosion of centrality values, making the network in this configuration extremely vulnerable to targeted attacks and computationally expensive to analyze.

![Centrality ](../results/plaw_raes/centrality_plot.png)

## Diffusion Dynamics

In this experiment, we evaluate the performance of the network by simulating a dynamic flooding process. Starting from a stabilized graph, a random source node generates a message. At each round, every informed node forwards the message to all its current neighbors, and the graph topology evolves according to the Power Law RAES model.

We define the flooding time as the number of rounds required for the message to reach all the nodes in the network.
The selected parameters are $n = 2^{15}, d=4, c = 3, k = 2.0$ and $p \in \{0.0,0.1,\ldots,0.9,1.0\}.$
As done before, we take the average of $100$ independent runs of $100$ rounds each.

![Flooding vs P](../results/plaw_raes/plot_flooding_vs_p_final.png)

We observe that for low values of $p,$ the flooding time is very low, settling around $3$ rounds.
This is expected due to the presence of strong hubs: once the message reaches a hub, which happens almost immediately due to their high degree, the hub forwards the information to thousands of nodes in a single step. We also notice that, as $p$ increases beyond $0.5,$ the flooding time rises monotonically, reaching $5.02$ rounds at $p=0.9.$ This shows that, as the probability of adopting a standard degree increases, the hubs become smaller and less frequent.
There is also a significant jump at the transition to the standard model, as the flooding time reaches $7$ rounds for $p=1.0$ due to the absence of hubs.
We explicitely note that the optimal performance is achieved around $p=0.2,$ suggesting that a small uniform component helps in reaching peripheral nodes. 